In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_5.py ──
"""
Shared infrastructure for MLFP03 Exercise 5 — Class Imbalance & Calibration.

Contains: Singapore credit scoring data loader, Kailash PreprocessingPipeline
wiring, cost-matrix constants, metric helpers, reliability-diagram helpers,
and an OUTPUT_DIR convention for every technique file to write visual proof
to the same place.

Technique-specific code (SMOTE call, focal loss gradient, Platt/Isotonic
model wiring) does NOT belong here — it lives in the per-technique files.
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY — every technique writes visual proof to the same place
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "ex5_imbalance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# BUSINESS CONTEXT — Singapore retail bank credit scoring
# ════════════════════════════════════════════════════════════════════════
# These constants drive every technique file. They are ILLUSTRATIVE round
# numbers for an unsecured personal-loan book, not figures from any bank:
#
#   FN (approve someone who defaults)  ≈ S$10,000 charged-off principal
#   FP (decline someone who would repay) ≈ S$1,500 forgone net interest
#        margin over the life of the loan (plus the lost relationship)
#
# A ~7:1 ratio puts the Bayes-optimal threshold t* = FP / (FP + FN) ≈ 0.13,
# close to the default rate — so the threshold genuinely matters. (A much
# cheaper FP, e.g. S$100, would make "decline almost everyone" optimal and
# no model could beat that trivial policy — check your cost numbers before
# you trust any threshold.)


@dataclass(frozen=True)
class CostMatrix:
    """Dollar cost of each confusion-matrix cell.

    fn = cost of missing a default (charge-off loss)
    fp = cost of a false decline (forgone interest margin + relationship)
    """

    fn: float = 10_000.0
    fp: float = 1_500.0

    @property
    def optimal_threshold(self) -> float:
        """Bayes-optimal threshold t* = fp / (fp + fn).

        Valid only for CALIBRATED probabilities from an UNWEIGHTED loss —
        a class-weighted model has already shifted its scores.
        """
        return self.fp / (self.fp + self.fn)

    def total_cost(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        return float(fp * self.fp + fn * self.fn)


DEFAULT_COSTS = CostMatrix(fn=10_000.0, fp=1_500.0)

# Annual application volume for ROI projections (illustrative).
ANNUAL_APPLICATIONS = 100_000


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring dataset
# ════════════════════════════════════════════════════════════════════════
# The dataset is loaded through the MLFPDataLoader so it works identically
# in local (.data_cache) and Colab (Drive + gdown) formats.

# Columns that MUST NOT be model inputs (Lesson 3.1 leakage rule):
#   customer_id              — a row identifier, not a property of the applicant
#   future_default_indicator — recorded AFTER the loan outcome is known; it
#                              agrees with ``default`` on ~99% of rows, so a
#                              model that sees it "predicts" default by
#                              reading the answer (Exercise 4 screens for it).
CREDIT_NON_FEATURE_COLUMNS: tuple[str, ...] = ("customer_id", "future_default_indicator")


def load_credit_splits(
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    """Load the SG credit scoring dataset and return (X_train, y_train, X_test, y_test, pos_rate).

    Holds out a stratified 20% test split FIRST, then fits kailash-ml's
    PreprocessingPipeline on the training rows only, so every technique file
    gets the same leak-free preprocessing. Returns numpy arrays ready for
    sklearn-style fit.
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet").drop(
        CREDIT_NON_FEATURE_COLUMNS
    )

    # Split FIRST, then fit imputation/encoding on the training rows only:
    # PreprocessingPipeline.setup() on the whole frame would fit them on the
    # test rows too (it splits only after fitting).
    result = split_then_preprocess(
        credit,
        target="default",
        test_size=0.2,
        seed=seed,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_cols = [c for c in result.train_data.columns if c != "default"]
    X_train, y_train, _ = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    pos_rate = float(y_train.mean())
    return X_train, y_train, X_test, y_test, pos_rate


# ════════════════════════════════════════════════════════════════════════
# METRIC HELPERS — complete taxonomy in one call
# ════════════════════════════════════════════════════════════════════════


def metrics_row(
    name: str,
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, Any]:
    """Compute a full metrics row for a given strategy name + probabilities.

    Returns a dict compatible with polars DataFrame construction so every
    technique file can build comparison tables with identical column shapes.
    """
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        "strategy": name,
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def print_metrics_table(rows: list[dict[str, Any]], title: str) -> None:
    """Print a comparison table across strategies."""
    print(f"\n{'=' * 70}")
    print(f"  {title}")
    print(f"{'=' * 70}")
    print(
        f"{'Strategy':<22} {'AUC-PR':>8} {'Brier':>8} {'F1':>8} "
        f"{'Precision':>10} {'Recall':>8}"
    )
    print("─" * 70)
    for r in rows:
        print(
            f"{r['strategy']:<22} {r['auc_pr']:>8.4f} {r['brier']:>8.4f} "
            f"{r['f1']:>8.4f} {r['precision']:>10.4f} {r['recall']:>8.4f}"
        )


def rows_to_dataframe(rows: list[dict[str, Any]]) -> pl.DataFrame:
    """Convert metrics rows to a polars DataFrame (for persistence / plots)."""
    return pl.DataFrame(rows)


# ════════════════════════════════════════════════════════════════════════
# RELIABILITY DIAGRAM DATA (calibration curve, binned)
# ════════════════════════════════════════════════════════════════════════


def reliability_bins(
    y_true: np.ndarray, y_proba: np.ndarray, n_bins: int = 10
) -> pl.DataFrame:
    """Compute a binned reliability diagram as a polars DataFrame.

    Columns: bin_lower, bin_upper, mean_pred, empirical_rate, count, gap
    """
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (y_proba >= lo) & (y_proba < hi if i < n_bins - 1 else y_proba <= hi)
        count = int(mask.sum())
        if count == 0:
            continue
        mean_pred = float(y_proba[mask].mean())
        empirical = float(y_true[mask].mean())
        rows.append(
            {
                "bin_lower": float(lo),
                "bin_upper": float(hi),
                "mean_pred": mean_pred,
                "empirical_rate": empirical,
                "count": count,
                "gap": float(abs(mean_pred - empirical)),
            }
        )
    return pl.DataFrame(rows)


def print_reliability(name: str, bins: pl.DataFrame) -> None:
    print(f"\n  Reliability bins — {name}")
    print(f"  {'mean_pred':>10} {'empirical':>10} {'|gap|':>8} {'n':>6}")
    print("  " + "─" * 38)
    for row in bins.iter_rows(named=True):
        print(
            f"  {row['mean_pred']:>10.3f} {row['empirical_rate']:>10.3f} "
            f"{row['gap']:>8.3f} {row['count']:>6}"
        )


# ════════════════════════════════════════════════════════════════════════
# BUSINESS ROI CALCULATOR
# ════════════════════════════════════════════════════════════════════════


def annual_roi(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float,
    costs: CostMatrix = DEFAULT_COSTS,
    annual_volume: int = ANNUAL_APPLICATIONS,
) -> dict[str, float]:
    """Project test-set confusion matrix onto an annual volume.

    Returns a dict with caught_defaults, missed_defaults, false_alarms,
    model_cost, no_model_cost, annual_savings — all in dollars.
    """
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    pos_rate = float(y_true.mean())
    scale = annual_volume / len(y_true)
    annual_fn = fn * scale
    annual_fp = fp * scale
    annual_tp = tp * scale
    n_defaults_annual = pos_rate * annual_volume
    model_cost = annual_fn * costs.fn + annual_fp * costs.fp
    no_model_cost = n_defaults_annual * costs.fn
    return {
        "threshold": float(threshold),
        "defaults_caught": float(annual_tp),
        "defaults_missed": float(annual_fn),
        "false_alarms": float(annual_fp),
        "model_cost_usd": float(model_cost),
        "no_model_cost_usd": float(no_model_cost),
        "annual_savings_usd": float(no_model_cost - model_cost),
        "annual_volume": int(annual_volume),
    }


def print_roi(name: str, roi: dict[str, float]) -> None:
    print(f"\n  ROI — {name}")
    print(f"    Threshold:          {roi['threshold']:.4f}")
    print(f"    Defaults caught:    {roi['defaults_caught']:>12,.0f}")
    print(f"    Defaults missed:    {roi['defaults_missed']:>12,.0f}")
    print(f"    False alarms:       {roi['false_alarms']:>12,.0f}")
    print(f"    Model cost:        S${roi['model_cost_usd']:>12,.0f}")
    print(f"    No-model cost:     S${roi['no_model_cost_usd']:>12,.0f}")
    print(f"    Annual savings:    S${roi['annual_savings_usd']:>12,.0f}")


# ════════════════════════════════════════════════════════════════════════
# PERSISTENCE — shared parquet of per-strategy probabilities
# ════════════════════════════════════════════════════════════════════════
# Each technique file writes its y_proba vector to a parquet under
# OUTPUT_DIR so that later technique files (threshold opt, calibration,
# final comparison) can read them without re-training.

PROBA_STORE = OUTPUT_DIR / "strategy_probabilities.parquet"


def save_strategy_proba(name: str, y_proba: np.ndarray) -> None:
    """Upsert a strategy's probability vector into the shared parquet store."""
    new_df = pl.DataFrame({"strategy": [name] * len(y_proba), "y_proba": y_proba})
    if PROBA_STORE.exists():
        existing = pl.read_parquet(PROBA_STORE)
        existing = existing.filter(pl.col("strategy") != name)
        combined = pl.concat([existing, new_df])
    else:
        combined = new_df
    combined.write_parquet(PROBA_STORE)


def load_strategy_proba(name: str) -> np.ndarray:
    """Read back a previously-saved probability vector for a strategy."""
    if not PROBA_STORE.exists():
        raise FileNotFoundError(
            f"{PROBA_STORE} not found — run the earlier technique files first."
        )
    df = pl.read_parquet(PROBA_STORE).filter(pl.col("strategy") == name)
    if df.height == 0:
        raise KeyError(f"Strategy {name!r} not found in {PROBA_STORE}")
    return df["y_proba"].to_numpy()


def list_saved_strategies() -> list[str]:
    if not PROBA_STORE.exists():
        return []
    df = pl.read_parquet(PROBA_STORE)
    return df["strategy"].unique().to_list()




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 5.5: Calibration (Platt + Isotonic) & Final Comparison
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - What calibration means: p_predicted = 0.2 means 20% default in reality
#   - Platt scaling (parametric logistic post-processing)
#   - Isotonic regression (non-parametric step-function post-processing)
#   - How to read a reliability diagram and spot under/over-confidence
#   - How to pick the production-ready strategy from all prior techniques
#
# PREREQUISITES: 01-04 in this directory (probabilities saved under
# outputs/ex5_imbalance/strategy_probabilities.parquet)
# ESTIMATED TIME: ~35 min
#
# 5-PHASE STRUCTURE:
#   Theory   — calibration intuition + why it matters for loan pricing
#   Build    — calibrate the cost-sensitive model with TrainingPipeline.calibrate
#   Train    — fit Platt and Isotonic variants
#   Visualise — reliability diagrams + final comparison table
#   Apply    — illustrative risk-based personal-loan pricing
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import lightgbm as lgb
import numpy as np
import plotly.graph_objects as go
import polars as pl
from dotenv import load_dotenv
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

from kailash_ml import TrainingPipeline


load_dotenv()



## THEORY — What calibration means and why loan pricing requires it


In [ ]:
# A model output p=0.2 is CALIBRATED if, among all applicants with
# p=0.2, exactly 20% actually default. A gradient booster trained with
# cost-sensitive weights can still be a good RANKER (AUC-PR) but is a poor
# CALIBRATOR: up-weighting defaulters tells the model defaults are more
# common than they are, so it systematically OVER-predicts default (you
# saw the mean predicted p jump well above the real rate in 5.2). Banking
# requires calibration for:
#
#   - LOAN PRICING: risk-based interest rates are computed as
#     rate = funding_cost + expected_loss(p) + margin. If p is
#     miscalibrated by 2x, the entire pricing curve is wrong.
#
#   - EXPECTED LOSS / ECL (IFRS 9): Stage 2 and Stage 3 provisions
#     are the sum of p*LGD*EAD over the portfolio. Miscalibrated p
#     directly misstates the provision line on the balance sheet.
#
#   - PORTFOLIO SIMULATION: stress tests model the loss distribution
#     by sampling from p. If p is uncalibrated, the 99th percentile
#     tail is wrong, and the CRO under-reserves.
#
# TWO POST-HOC CALIBRATORS:
#
#   PLATT SCALING — fits a 2-parameter logistic function
#       p_cal = 1 / (1 + exp(A * raw + B))
#     Parametric, low variance, works well with small calibration sets.
#     Assumes the miscalibration is sigmoid-shaped.
#
#   ISOTONIC REGRESSION — fits a non-decreasing step function.
#     Non-parametric, higher variance, more flexible. Needs >1000
#     calibration samples to avoid overfitting. It can correct ANY
#     monotone distortion of the scores (not just a sigmoid-shaped one),
#     but because it is monotone by construction it cannot fix a
#     non-monotone one — and it never changes the ranking (AUC stays put,
#     apart from ties created by its flat steps).
#
#   Both are fitted on a held-out CALIBRATION slice (20% of the training
#   rows the booster never saw): calibrating on the same rows the booster
#   trained on would learn its over-confidence on training data, not its
#   behaviour on new applicants.
#
# RULE OF THUMB: small calibration set -> Platt; large -> Isotonic;
# always check with a reliability diagram.



## BUILD + TRAIN — wrap the cost-sensitive LightGBM in two calibrators


In [ ]:
X_train, y_train, X_test, y_test, pos_rate = load_credit_splits()

scale_weight = (1 - pos_rate) / pos_rate
# 80% of the training rows fit the weighted booster; the other 20% are
# kept back to fit the calibration maps.
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)
base_estimator = lgb.LGBMClassifier(
    n_estimators=300,
    scale_pos_weight=scale_weight,
    random_state=42,
    verbose=-1,
)
base_estimator.fit(X_fit, y_fit)

# kailash-ml's TrainingPipeline.calibrate wraps an already-fitted model
# (frozen — it is not re-trained) and fits the calibration map on the
# held-out rows. No registry is needed for calibration alone.
pipeline = TrainingPipeline(feature_store=None, registry=None)
X_cal_frame = pl.DataFrame(X_cal, schema=[f"f{i}" for i in range(X_cal.shape[1])], orient="row")
y_cal_series = pl.Series("default", y_cal)

print("\n" + "=" * 70)
print("  Exercise 5.5 — Calibration (Platt + Isotonic)")
print("=" * 70)
print(f"  Fitting Platt scaling on {len(y_cal):,} held-out rows...")
platt = asyncio.run(
    pipeline.calibrate(base_estimator, X_cal_frame, y_cal_series, method="sigmoid")
)
print(f"  Fitting Isotonic regression on {len(y_cal):,} held-out rows...")
isotonic = asyncio.run(
    pipeline.calibrate(base_estimator, X_cal_frame, y_cal_series, method="isotonic")
)

y_proba_platt = platt.predict_proba(X_test)[:, 1]
y_proba_iso = isotonic.predict_proba(X_test)[:, 1]
save_strategy_proba("platt_calibrated", y_proba_platt)
save_strategy_proba("isotonic_calibrated", y_proba_iso)



### Checkpoint 5


In [ ]:
brier_raw = brier_score_loss(y_test, load_strategy_proba("cost_sensitive_scale"))
assert 0 <= y_proba_platt.min() and y_proba_platt.max() <= 1, "Platt out of range"
assert 0 <= y_proba_iso.min() and y_proba_iso.max() <= 1, "Isotonic out of range"
assert brier_score_loss(y_test, y_proba_platt) < brier_raw, "Platt must improve Brier"
assert brier_score_loss(y_test, y_proba_iso) < brier_raw, "Isotonic must improve Brier"
print("[ok] Checkpoint 5 — both calibrators improve Brier over the raw weighted model\n")



## VISUALISE — reliability diagrams + final strategy comparison


ECE = count-weighted mean |mean_pred - empirical_rate| over bins.


In [ ]:
# Reliability bins for each variant
bins_baseline = reliability_bins(y_test, load_strategy_proba("baseline"))
bins_cost = reliability_bins(y_test, load_strategy_proba("cost_sensitive_scale"))
bins_platt = reliability_bins(y_test, y_proba_platt)
bins_iso = reliability_bins(y_test, y_proba_iso)

print_reliability("Baseline", bins_baseline)
print_reliability("Cost-sensitive", bins_cost)
print_reliability("Platt", bins_platt)
print_reliability("Isotonic", bins_iso)



def expected_calibration_error(bins: pl.DataFrame) -> float:
    return float((bins["count"] * bins["gap"]).sum() / bins["count"].sum())


y_proba_cost = load_strategy_proba("cost_sensitive_scale")
calibration_summary = [
    (name, proba, expected_calibration_error(bins))
    for name, proba, bins in [
        ("Baseline", load_strategy_proba("baseline"), bins_baseline),
        ("Cost-sensitive", y_proba_cost, bins_cost),
        ("Platt", y_proba_platt, bins_platt),
        ("Isotonic", y_proba_iso, bins_iso),
    ]
]
print(f"\n  Real default rate in test: {y_test.mean():.3f}")
print(f"  {'Variant':<16} {'mean p':>8} {'ECE':>8} {'Brier':>8}")
for name, proba, ece in calibration_summary:
    brier = metrics_row(name, y_test, proba)["brier"]
    print(f"  {name:<16} {proba.mean():>8.3f} {ece:>8.4f} {brier:>8.4f}")
# INTERPRETATION: A point ABOVE the diagonal in a reliability bin means
# the model's probabilities there are too LOW (more defaults happen than
# predicted); BELOW means too HIGH. Read the table with that in mind:
# where does the cost-sensitive model's mean p sit relative to the real
# default rate, and how much do Platt and Isotonic shrink ECE and Brier?
# (Brier mixes calibration with discrimination; ECE isolates calibration.)

# Final comparison table across every strategy we've trained
strategies = [
    ("Baseline (none)", "baseline"),
    ("SMOTE", "smote"),
    ("Cost-sens (scale)", "cost_sensitive_scale"),
    ("Cost-sens (matrix)", "cost_sensitive_matrix"),
    ("Focal gamma=2.0", "focal_gamma_2.0"),
    ("Cost + Platt", "platt_calibrated"),
    ("Cost + Isotonic", "isotonic_calibrated"),
]
# Every strategy must exist — run 01-04 first. A missing one raises with
# a clear message rather than silently shrinking the comparison.
all_rows: list[dict] = []
for display, key in strategies:
    p = load_strategy_proba(key)
    all_rows.append(metrics_row(display, y_test, p))

print_metrics_table(all_rows, "FINAL COMPARISON — all imbalance strategies")

final_df = pl.DataFrame(all_rows)
final_df.write_parquet(OUTPUT_DIR / "final_comparison.parquet")
print(f"\n  Saved: {OUTPUT_DIR / 'final_comparison.parquet'}")



### Visual: Reliability diagram (calibration curves)


In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Perfect calibration",
        line=dict(dash="dash", color="#9ca3af", width=1),
    )
)
for label, bins in [
    ("Baseline", bins_baseline),
    ("Cost-sensitive", bins_cost),
    ("Platt", bins_platt),
    ("Isotonic", bins_iso),
]:
    # bins is a polars DataFrame with columns mean_pred / empirical_rate /
    # count (see shared.mlfp03.ex_5.reliability_bins). Filter out empty
    # bins, then extract the two scatter columns.
    nonempty = bins.filter(pl.col("count") > 0)
    mean_pred = nonempty["mean_pred"].to_list()
    frac_pos = nonempty["empirical_rate"].to_list()
    fig.add_trace(go.Scatter(x=mean_pred, y=frac_pos, mode="lines+markers", name=label))
fig.update_layout(
    title="Reliability Diagram: predicted probability vs observed default rate",
    xaxis_title="Mean predicted probability",
    yaxis_title="Fraction of positives (actual default rate)",
    height=500,
    legend=dict(orientation="h", y=-0.2),
)
viz_path = OUTPUT_DIR / "ex5_05_reliability_diagram.html"
fig.write_html(str(viz_path))
print(f"  Saved: {viz_path}")



### Visual: Brier score comparison bar chart


In [ ]:
fig2 = go.Figure()
fig2.add_trace(
    go.Bar(
        x=[r["strategy"] for r in all_rows],
        y=[r["brier"] for r in all_rows],
        marker_color=[
            (
                "#10b981"
                if r["brier"] == min(rr["brier"] for rr in all_rows)
                else "#6366f1"
            )
            for r in all_rows
        ],
        text=[f"{r['brier']:.4f}" for r in all_rows],
        textposition="outside",
    )
)
fig2.update_layout(
    title="Brier Score Comparison: lower = better calibration (green = best)",
    xaxis_title="Strategy",
    yaxis_title="Brier score",
    height=450,
)
viz_path2 = OUTPUT_DIR / "ex5_05_brier_comparison.html"
fig2.write_html(str(viz_path2))
print(f"  Saved: {viz_path2}")

# Pick the production-ready strategy
best_auc_pr = max(all_rows, key=lambda r: r["auc_pr"])
best_brier = min(all_rows, key=lambda r: r["brier"])
print(
    f"\n  Best AUC-PR: {best_auc_pr['strategy']} "
    f"(AUC-PR={best_auc_pr['auc_pr']:.4f})"
)
print(f"  Best Brier:  {best_brier['strategy']} (Brier={best_brier['brier']:.4f})")



## APPLY — Risk-based personal-loan pricing (illustrative)


In [ ]:
# SCENARIO (illustrative): a Singapore retail bank prices every personal
# loan individually with a risk-based formula:
#
#     APR = funding_cost + expected_loss(p) + operating_margin
#     expected_loss(p) = p * LGD * EAD
#
# If p is systematically too high (as with the raw weighted model), every
# good customer is over-priced and drifts to competitors; if too low, the
# risky book is under-priced and provisions are understated. Model-risk
# reviewers therefore ask to see calibration on held-out data (the
# reliability diagram and ECE above) before a pricing model goes live.
#
# Production recipe:
#   1. Train a strong ranker (LightGBM, optionally class-weighted)
#   2. Post-calibrate on held-out data (Isotonic with plenty of data,
#      Platt with little)
#   3. Apply the Bayes threshold t* = cost_FP / (cost_FP + cost_FN) to
#      the CALIBRATED probabilities
#   4. Price loans from the calibrated p
#   5. Monitor drift (Exercise 8)
#
# The ROI below applies t* to the raw weighted scores and to the two
# calibrated versions — t* is only valid for the calibrated ones.

t_star = DEFAULT_COSTS.optimal_threshold
roi_by_variant: dict[str, dict] = {}
for display, proba in [
    ("Cost-sens (raw)", y_proba_cost),
    ("Cost + Platt", y_proba_platt),
    ("Cost + Isotonic", y_proba_iso),
]:
    roi = annual_roi(y_test, proba, threshold=t_star, annual_volume=ANNUAL_APPLICATIONS)
    roi_by_variant[display] = roi
    print_roi(f"{display} @ t*={t_star:.4f}", roi)

best_variant = max(roi_by_variant, key=lambda k: roi_by_variant[k]["annual_savings_usd"])
print(f"\n  Highest annual savings at t*: {best_variant}")



## PRODUCTION RECOMMENDATION — computed from this run


From this run (~{y_test.mean():.0%} default rate, cost ratio
  {DEFAULT_COSTS.fn / DEFAULT_COSTS.fp:.1f}:1):

    1. Best ranking (AUC-PR):     {best_auc_pr['strategy']}
    2. Best Brier:                {best_brier['strategy']}
    3. Best calibrated variant:   {best_calibrated[0]} (ECE {best_calibrated[2]:.4f})
    4. Decision rule:             decline if calibrated p >= t* = {t_star:.3f}
    5. Report AUC-PR + Brier/ECE + annual S$ savings to the risk committee

  If you use SMOTE or class weights, recalibrate on held-out data
  before reading the outputs as probabilities.


In [ ]:
best_calibrated = min(calibration_summary[2:], key=lambda t: t[2])
print(
)



## DESTINATION-FIRST CLOSE — km.diagnose


In [ ]:
# This lesson built calibration from primitives — Platt scaling, isotonic
# regression, cost-aware threshold optimisation. The kailash-ml SDK
# packages the diagnostic surface (per-class metrics, class-balance
# severity, confusion matrix) into a single call.
#
# Destination-first: when the journey is internalised, the SDK is one line.

from kailash_ml import diagnose

# `kind="classical_classifier"` dispatches to the sklearn ClassifierMixin
# adapter. TrainingPipeline.calibrate returns an sklearn
# CalibratedClassifierCV, which implements the ClassifierMixin interface.
# Use the isotonic variant — typically the better calibrator for >1k samples.
report = diagnose(
    isotonic, kind="classical_classifier", data=(X_test, y_test), show=False
)
print()
print("  km.diagnose model    : Isotonic-calibrated LightGBM")
print(f"  km.diagnose metrics  : {report.metrics}")
print(f"  km.diagnose severity : {report.severity}")
print()
print("km.diagnose: 1 call -> the same diagnostic surface the lesson body")
print("hand-rolled across Platt + Isotonic. Destination-first: when the")
print("journey is internalised, the SDK is one line.")



## REFLECTION


[x] Platt scaling: parametric logistic post-calibration
  [x] Isotonic regression: non-parametric step-function post-calibration
  [x] Read reliability diagrams to spot under/over-confidence
  [x] Final comparison across all seven strategies (baseline, SMOTE,
      cost-sens x2, focal, Platt, Isotonic)
  [x] Translated the winner into annual S$ savings for loan pricing
  [x] Documented the production recipe for Singapore consumer credit

  WHOLE-EXERCISE INSIGHT: The winning strategy on financial tabular
  data is almost never "the clever paper trick." It's cost-sensitive
  learning + post-hoc calibration + cost-matrix threshold tuning.
  Simple, production-grade, auditable.

  NEXT: Exercise 6 adds SHAP interpretability — the per-decision
  explanations that model-risk reviewers and customers ask for.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — 5.5 (and Exercise 5 as a whole)")
print("=" * 70)
print(
)

